In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()

xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()
dids = df['did'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/vlos/*'))
#files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20240428T031503_V202602220014_0444280501.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20241025T081002_V202602220112_0450250501.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20250428T200004_V202602220258_0544280511.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20250920T031503_V202602220437_0549200501.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20251007T124003_V202602220437_0550070502.fits.gz']

In [9]:
file = files[-1]

with fits.open(file) as hdul:
    header = hdul[0].header
    data = hdul[0].data

did = int(file.split('_')[-1].split('.')[0])
data = undistort(data, header, xd, yd)

view = View.from_header(header)
view.update(crota=view.crota + 0.25, xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
            inplace=True)

view_ = view.update(crota=180, increment=True)

In [10]:
view_new = view.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=0 * 3600)

show_data(view_new.reproject(data, view), view_new, label=r'$V_{los}, km/s$', cmap='seismic', vmin=-2, vmax=2)

In [11]:
V = (view.velocity(mu_thr=0., cbs=False) - view_.velocity(mu_thr=0., cbs=False)) / 2 / 1000

plt.figure(figsize=(10,10))
plt.imshow(V, 'seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [12]:
V_ = (data.copy() - view_.reproject(data, view)) / 2

plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [16]:
t = ~np.isnan(V)
x, y = V[t], V_[t]
t = np.where(np.abs(y - x) < 0.5)
x, y = x[t], y[t]

k = np.nanmean(x * y) / np.nanmean(x ** 2)
print(k)

plt.figure(figsize=(10,10))
plt.plot([-2,2], [-2,2], color='black', lw=0.5)
plt.scatter(V, V_, s=0.001)
plt.plot([-2,2], [-2 * k, 2 * k], '--', color='black', lw=1)

plt.xlabel('Velocity, km/s')
plt.ylabel('Doppler velocity, km/s')

plt.xlim(-2,2)
plt.ylim(-2,2)

plt.grid(True)

plt.tight_layout()

0.9491603962483086


In [18]:
V = view.velocity(mu_thr=0., cbs=False) / 1000
V -= np.nanmean(V)

V_ = data.copy()
V_[np.isnan(V)] = np.nan
V_ -= np.nanmean(V_)

U = V_ - V * k - 0.5

In [19]:
show_data(view_new.reproject(U, view), view_new, label=r'$V_{los}, km/s$', cmap='seismic', vmin=-1, vmax=1)

In [20]:
mu = view.mu()

t = np.where(mu > 0.1)
p = np.polyfit(mu[t], U[t], 3)

dx = 0.1
x = np.arange(dx / 2, 1, dx)
y = []
for x_ in x:
    t = np.where(np.abs(mu - x_) < dx / 2)
    y += [np.nanmean(U[t])]
y = np.array(y)

plt.figure(figsize=(10,10))
plt.scatter(mu, U, s=0.01)
#plt.plot(x, y, '--', color='black')
plt.plot(np.arange(0,1.01,0.01), np.polyval(p, np.arange(0,1.01,0.01)), color='black', lw=1.25)

plt.title('Convective blueshift')
plt.xlabel(r'$\mu$')
plt.ylabel('Doppler velocity, km/s')

plt.xlim(0,1)
plt.ylim(-1.,0.2)

plt.gca().invert_xaxis()

plt.grid(True)
plt.tight_layout()

In [21]:
U_ = np.polyval(p, mu)
S = U - U_

show_data(view_new.reproject(S, view), view_new, label=r'$V_{los}, km/s$', cmap='seismic', vmin=-1, vmax=1)

In [22]:
header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    2 / number of data axis                            
NAXIS1  =                  896 / length of data axis 1                          
NAXIS2  =                  896 / length of data axis 2                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
LONGSTRN= 'OGIP 1.0'           / The HEASARC Long String Convention may be used.
COMMENT   This FITS file may contain long string keyword values that are        
COMMENT   continued over multiple keywords.  The HEASARC convention uses the &  
COMMENT   character at the end of each substring which is then continued        
COMMENT   on the next keywor